# pyBKT comparison on Colab

AdaptAid evaluates its own Bayesian Knowledge Tracing against **pyBKT** (Badrinath et al., 2021) on ASSISTments 2009. pyBKT's fast C++ engine needs OpenMP: it compiles on Linux (Colab) but fell back to slow pure Python on the project Mac. This notebook produces pyBKT's predictions with **exactly the same cleaned data, skills and student folds** as `research/src/bkt_eval.py`.

**Steps:** Runtime → Run all (a CPU runtime is enough). At the end, a file `pybkt_predictions.csv` downloads. Put it in `research/outputs/` of the project and run:

```bash
backend/.venv/bin/python -m research.src.bkt_eval --pybkt-csv research/outputs/pybkt_predictions.csv
```

## 1. An isolated Python 3.11 environment with the versions pyBKT needs

pyBKT 1.4.3 works only with numpy 1.26, pandas 2.2 and scikit-learn 1.5. `uv` builds a separate environment so Colab's own packages are untouched (no runtime restart needed). pyBKT is compiled from source here, which gives the C++ engine.

In [ ]:
!pip -q install uv
!uv python install 3.11 -q
!uv venv -q -p 3.11 /content/bkt
!uv pip install -q -p /content/bkt/bin/python numpy==1.26.4 pandas==2.2.3 scikit-learn==1.5.2 setuptools wheel
!uv pip install -q -p /content/bkt/bin/python --no-build-isolation pyBKT==1.4.3

In [ ]:
# Check that the C++ engine was built (compiled .so files inside the pyBKT package).
!/content/bkt/bin/python -c "import pathlib, pyBKT; so = sorted(p.name for p in pathlib.Path(pyBKT.__file__).parent.rglob('*.so')); print('C++ engine active:', bool(so), so)"

## 2. The evaluation script (same cleaning, skills and folds as the project)

In [ ]:
%%writefile /content/pybkt_eval.py
"""pyBKT predictions on ASSISTments 2009 with the exact data, skills and folds of
research/src/bkt_eval.py, so the two models are scored on identical predictions."""
import argparse
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from pyBKT.models import Model

URL = "https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/as.csv"
EVAL_SKILLS = 40  # must equal --skills in research/src/bkt_eval.py (the student split depends on it)


def load() -> pd.DataFrame:
    raw = pd.read_csv(URL, encoding="latin-1", low_memory=False,
                      usecols=["order_id", "user_id", "skill_id", "skill_name", "correct", "original"])
    df = raw[raw["original"] == 1].dropna(subset=["skill_name"])
    df = df.drop_duplicates(subset="order_id", keep="first").sort_values(["user_id", "order_id"])
    df["correct"] = df["correct"].astype(int).clip(0, 1)
    return df[["order_id", "user_id", "skill_name", "correct"]].reset_index(drop=True)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--skills", type=int, default=40, help="fit pyBKT on the N largest skills")
    ap.add_argument("--out", default="/content/pybkt_predictions.csv")
    args = ap.parse_args()
    t0 = time.time()
    df = load()
    print(f"{len(df):,} answers after cleaning (expected 251,405)")
    counts = df.groupby("skill_name")["user_id"].nunique()
    order = sorted(counts.index, key=lambda s: (-int(counts[s]), s))
    df = df[df["skill_name"].isin(order[:EVAL_SKILLS])]
    fit_skills = set(order[: args.skills])
    users = np.array(sorted(df["user_id"].unique()))
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    parts = []
    for fold, (tr, te) in enumerate(kf.split(users)):
        tr_u, te_u = set(users[tr]), set(users[te])
        train = df[df["user_id"].isin(tr_u) & df["skill_name"].isin(fit_skills)]
        test = df[df["user_id"].isin(te_u) & df["skill_name"].isin(fit_skills)]
        model = Model(seed=42, num_fits=3, parallel=True)
        model.fit(data=train)
        pred = model.predict(data=test)
        parts.append(pd.DataFrame({"fold": fold, "order_id": pred["order_id"].astype(int),
                                   "skill_name": pred["skill_name"], "pybkt": pred["correct_predictions"]}))
        print(f"fold {fold + 1}/5 done ({time.time() - t0:.0f}s)", flush=True)
    out = pd.concat(parts)
    out.to_csv(args.out, index=False)
    print(f"wrote {len(out):,} predictions for {out['skill_name'].nunique()} skills to {args.out}")


if __name__ == "__main__":
    main()


## 3. Run it

With the C++ engine, all 40 skills take a few minutes. If the check above printed `C++ engine active: False`, change `--skills 40` to `--skills 10`.

In [ ]:
!/content/bkt/bin/python /content/pybkt_eval.py --skills 40

## 4. Download the predictions

In [ ]:
from google.colab import files

files.download('/content/pybkt_predictions.csv')